# Module 6 - Lab 3: Two Sensors in One Recording

A phone can record several sensors at the same time. This lab reads such a recording - an accelerometer and a gyroscope captured during the same drive - and shows both together.

- What did each sensor measure, and how densely?
- Do the two sensors need to be aligned in time, as the two phones in Lab 2 did?
- Where do both sensors react to the same event, and where does only one of them react?
- Which results are trustworthy, and which does the recording itself rule out?

## Learning goals
- Load a measurement file that contains more than one sensor.
- Tell apart two reasons for differing timestamps: separate clocks and separate sampling.
- Display quantities with different units meaningfully on a shared time axis.
- Recognise from the data whether an accelerometer recording still contains gravity.
- Judge which conclusions a recording supports before reporting numbers from it.

<div style="border: 3px solid #b45309; background: #fff7ed; padding: 18px 20px; margin: 16px 0 24px 0; border-radius: 8px;">
<h2 style="margin-top: 0; color: #9a3412;">One recording, several sensors</h2>
<p>This lab expects <strong>one file containing at least two sensor sheets</strong>, as phyphox writes it when several sensors are recorded together. The sensors are recognised by the <strong>units</strong> in their columns, so the language of the app does not matter.</p>
<p>Unlike Lab 2, nothing has to be aligned here: all sensors were started by the same recording and therefore already share one clock.</p>
</div>

## Section 1: Import Libraries

In [ ]:
# Section 1: Import Libraries
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from metadata_loader import load_metadata_context
from multi_sensor import (
    check_gravity,
    check_shared_clock,
    compare_sensor_activity,
    compare_sensor_summaries,
    compare_time_quality,
    load_multi_sensor_file,
    plot_multi_sensor_axes,
    plot_multi_sensor_overview,
    plot_sensor_activity_relation,
    plotly_multi_sensor_explorer,
    resample_sensors_to_common_grid,
    summarize_multi_sensor_file,
)

pd.set_option('display.max_columns', 30)
pd.set_option('display.precision', 4)

print('Libraries imported successfully.')

## Section 2: Choose the Recording

Enter the multi-sensor file. Every sheet whose columns carry a recognised unit becomes one sensor; sheets holding device or timing metadata are skipped.

In [ ]:
# Section 2: Select the multi-sensor file
# ============================== YOUR INPUT ==================================
# One file containing at least two sensor sheets.
multi_sensor_path = 'data/suspension/Multi-Sensor/AccGyro.xls'
# ============================================================================

In [ ]:
# Section 2: Load every sensor of the recording
metadata = load_metadata_context(project_root)['public_metadata']

sensor_bundle = load_multi_sensor_file(multi_sensor_path, metadata, project_root=project_root)

print('Sheets in the file :', sensor_bundle['sheet_names'])
print('Skipped as metadata:', sensor_bundle['skipped_sheets'])
display(summarize_multi_sensor_file(sensor_bundle))

### Observation 1: What Was Recorded

- Which sensors does the file contain, and for how long?
- Was the magnitude recorded, or was it computed from the axes here?
-

## Section 3: Do the Sensors Need Aligning?

In Lab 2 two phones had to be shifted against each other, because each was started by hand and each carried its own clock. Here the situation is different: all sensors belong to one recording and were started by it, so they already share a time axis.

What still differs is *when* each sensor delivered its samples. That is a property of the sensors, not of the clock, and it does not call for an offset.

In [ ]:
# Section 3: Check the shared clock
display(check_shared_clock(sensor_bundle))

In [ ]:
# Section 3: Time step quality per sensor
display(compare_time_quality(sensor_bundle))

## Section 4: Both Sensors on a Shared Time Axis

Acceleration and rotation have different units, so they cannot share one y axis. They can share the x axis, which is what matters here: it makes visible which events both sensors saw.

In [ ]:
# Section 4: Magnitude of every sensor above one another
plot_multi_sensor_overview(sensor_bundle)

In [ ]:
# Section 4: Every axis of every sensor
plot_multi_sensor_axes(sensor_bundle)

In [ ]:
# Section 4: Inspect both sensors interactively (zoom, hover, toggle traces)
plotly_multi_sensor_explorer(sensor_bundle)

### Observation 2: Shared Events

Zoom into a moment where one sensor shows a clear peak.

- Does the other sensor react at the same moment?
- Can you find an event that only one of the two sensors sees? What could that be?
-

## Section 5: How Closely Do the Sensors Track Each Other?

Their values cannot be compared - m/s^2 against rad/s is meaningless. What can be compared is *when* each sensor is active, because a bump, a braking manoeuvre, or a corner shows up in both.

To relate them sample by sample, both are put on a shared grid, since they sampled at different moments. **This interpolation is used for this comparison only**; every per-sensor result stays on the originally recorded samples.

In [ ]:
# Section 5: Activity comparison between the sensors
display(compare_sensor_activity(sensor_bundle))
plot_sensor_activity_relation(sensor_bundle)

The activity correlation asks whether both sensors are busy at the same times. The value correlation compares the raw values and is expected to be near zero - the two quantities have no reason to rise and fall together.

### Observation 3: Relationship Between the Sensors

- How strong is the activity correlation, and what does that tell you about the drive?
- Would you expect a stronger relationship? What in the driving would produce one?
-

## Section 6: What This Recording Does and Does Not Support

phyphox can record acceleration with or without gravity. On a plot of a driving car the difference is barely visible, but it decides whether integrating the signal to a speed means anything at all. So it is checked against the data rather than assumed.

In [ ]:
# Section 6: Does the accelerometer signal still contain gravity?
display(check_gravity(sensor_bundle))

In [ ]:
# Section 6: Mode-specific results per sensor
display(compare_sensor_summaries(sensor_bundle))

### Observation 4: Trustworthy and Untrustworthy Results

Look at the `note` column in the table above before you read any number from it.

- Which values does gravity make unreliable?
- The reported speed may well look plausible. Why is that more dangerous than an obviously wrong value?
- Which recording would you have to make to obtain a trustworthy speed?
-

## Section 7: Document Your Comparison

**What the recording contains:**

**Why no time alignment was needed:**

**Where both sensors saw the same events:**

**Which results I would report, and which I would not:**

**What I would record differently next time:**